8E.1: LSTM RUL Predictor

CELL 1 — Setup, Load Data & Sequence Preparation

### 1. Setup, Load Data & Sequence Preparation
**Tujuan:** Memuat data RUL track, filter WARNING+CRITICAL, lalu reshape data ke format 3D sequence untuk LSTM: (samples, SEQ_LEN=24, n_features=69).  
**Input:** data/processed/X_train_rul.parquet, y_train_rul.parquet, X_val_rul, y_val_rul, X_test_rul, y_test_rul, df_ssbs_rul.parquet  
**Output:** X_train_seq (shape: N×24×69), X_val_seq, X_test_seq, y_train_seq, y_val_seq, y_test_seq  
**Catatan:** SEQ_LEN=24 timesteps = 24 jam look-back window. [WARN] TensorFlow mengalami DLL load error pada hardware ini (CPU tanpa AVX/AVX2). Model V2 sudah tersimpan sebagai checkpoint  jalankan Cell 2 untuk load checkpoint langsung.  


In [1]:
# FASE 8E — Cell 1: Setup, Load Data & Sequence Preparation
import sys, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns, joblib
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

NOTEBOOK_DIR = Path().resolve()
ML_ROOT      = NOTEBOOK_DIR.parent.parent
SRC_PATH     = ML_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from config import DATA_PROCESSED_DIR, MODELS_DL_DIR, GLOBAL_SEED

np.random.seed(GLOBAL_SEED)
tf.random.set_seed(GLOBAL_SEED)

SEP = "=" * 65
sep = "-" * 65

print(SEP)
print("  FASE 8E — LSTM RUL PREDICTOR (WARNING + CRITICAL ONLY)")
print(SEP)

# ────────────────────────────────────────────────────────────
# LOAD DATA FULL
# ────────────────────────────────────────────────────────────
X_train_full = pd.read_parquet(DATA_PROCESSED_DIR / "X_train_rul.parquet")
y_train_full = pd.read_parquet(DATA_PROCESSED_DIR / "y_train_rul.parquet").squeeze()
X_val_full   = pd.read_parquet(DATA_PROCESSED_DIR / "X_val_rul.parquet")
y_val_full   = pd.read_parquet(DATA_PROCESSED_DIR / "y_val_rul.parquet").squeeze()
X_test_full  = pd.read_parquet(DATA_PROCESSED_DIR / "X_test_rul.parquet")
y_test_full  = pd.read_parquet(DATA_PROCESSED_DIR / "y_test_rul.parquet").squeeze()

df_ssbs_rul = pd.read_parquet(DATA_PROCESSED_DIR / "df_ssbs_rul.parquet")
df_ssbs_rul = df_ssbs_rul.sort_values(
    ["machine_id", "timestamp"]).reset_index(drop=True)

print(f"\n  Data full berhasil dimuat.")

# ────────────────────────────────────────────────────────────
# DEFINISI MESIN
# ────────────────────────────────────────────────────────────
TRAIN_MACHINES = ["M-01","M-02","M-03","M-04","M-05",
                  "M-06","M-07","M-08","M-09","M-10",
                  "M-11","M-12","M-13","M-14"]
VAL_MACHINES   = ["M-15","M-16","M-17"]
TEST_MACHINES  = ["M-18","M-19","M-20"]

# ────────────────────────────────────────────────────────────
# FILTER WARNING + CRITICAL (identik dengan 08D)
# ────────────────────────────────────────────────────────────
def filter_warn_crit(X_full, y_full, machines):
    df_labels = (
        df_ssbs_rul[df_ssbs_rul["machine_id"].isin(machines)]
        .reset_index(drop=True)
    )
    mask = df_labels["health_label_encoded"].isin([1, 2]).values
    X_filt = X_full[mask].reset_index(drop=True)
    y_filt = y_full[mask].reset_index(drop=True)
    return X_filt, y_filt

X_train, y_train = filter_warn_crit(X_train_full, y_train_full, TRAIN_MACHINES)
X_val,   y_val   = filter_warn_crit(X_val_full,   y_val_full,   VAL_MACHINES)
X_test,  y_test  = filter_warn_crit(X_test_full,  y_test_full,  TEST_MACHINES)

print(f"\n  Shape setelah filter WARNING+CRITICAL:")
print(f"    {'Split':<10} {'X Shape':>15} {'y Shape':>10}")
print(f"    {'-'*38}")
for name, X, y in [("Train", X_train, y_train),
                   ("Val",   X_val,   y_val),
                   ("Test",  X_test,  y_test)]:
    print(f"    {name:<10} {str(X.shape):>15} {str(y.shape):>10}")

# ────────────────────────────────────────────────────────────
# SEQUENCE PREPARATION
# ────────────────────────────────────────────────────────────
SEQ_LEN = 24  # 24 timesteps = 24 jam look-back

def create_sequences(X, y, seq_len):
    sequences, targets = [], []
    for i in range(seq_len, len(X)):
        sequences.append(X[i - seq_len:i])
        targets.append(y[i])
    return np.array(sequences), np.array(targets)

X_train_seq, y_train_seq = create_sequences(X_train.values, y_train.values, SEQ_LEN)
X_val_seq,   y_val_seq   = create_sequences(X_val.values,   y_val.values,   SEQ_LEN)
X_test_seq,  y_test_seq  = create_sequences(X_test.values,  y_test.values,  SEQ_LEN)

print(f"\n{sep}")
print("  SEQUENCE PREPARATION — Reshape ke 3D")
print(sep)
print(f"\n  SEQ_LEN (look-back window) : {SEQ_LEN} timesteps")
print(f"\n  {'Split':<10} {'Before (2D)':>18} {'After (3D)':>25}")
print(f"  {'-'*55}")
for name, X_f, X_s in [("Train", X_train, X_train_seq),
                         ("Val",   X_val,   X_val_seq),
                         ("Test",  X_test,  X_test_seq)]:
    print(f"  {name:<10} {str(X_f.shape):>18} {str(X_s.shape):>25}")

print(f"\n  Input shape untuk LSTM: "
      f"(samples, {SEQ_LEN}, {X_train_seq.shape[2]})")

print(f"\n{sep}")
print("  STATISTIK TARGET y_seq (rul_days)")
print(sep)
print(f"\n  {'Split':<12} {'Min':>8} {'Max':>8} {'Mean':>8} {'Median':>8}")
print(f"  {'-'*47}")
for name, y in [("y_train_seq", y_train_seq),
                ("y_val_seq",   y_val_seq),
                ("y_test_seq",  y_test_seq)]:
    print(f"  {name:<12} {y.min():>8.2f} {y.max():>8.2f} "
          f"{y.mean():>8.2f} {np.median(y):>8.2f}")

print(f"\n{SEP}")
print("  [OK] Cell 1 selesai — lanjut ke Cell 2 untuk training.")



[TensorFlow DLL Diagnostic] Analyzing: c:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\venv\lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "c:\PORTFOLIO\PROJECTS\PBL\Predictive Maintenance\projects\predictive-maintenance-monorepo\machine_learning\venv\lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

Checkpoint V2

### 2. Load Checkpoint V2 (Val MAE = 0.8160)
**Tujuan:** Memuat model LSTM V2 terbaik dari checkpoint yang sudah tersimpan sebelumnya (Val MAE = 0.8160 hari di epoch 184).  
**Input:** models/dl_track/lstm_rul_best_v2.keras  
**Output:** model_v2  LSTM model siap untuk evaluasi (47.649 params)  
**Catatan:** Checkpoint V2 merupakan hasil training 200 epoch dengan L2 regularization dan Dropout 0.3. JALANKAN CELL INI (bukan Cell 3 training) jika ingin menampilkan evaluasi tanpa re-training.  


In [ ]:
# CELL 2 — Load Best Checkpoint V2 (Val MAE = 0.8160)
from tensorflow.keras.models import load_model

print("=" * 65)
print("  LOAD CHECKPOINT V2 TERBAIK (Val MAE = 0.8160)")
print("=" * 65)

checkpoint_v2_path = str(MODELS_DL_DIR / "lstm_rul_best_v2.keras")
model_v2 = load_model(checkpoint_v2_path)

# Alias untuk Cell 4
model = model_v2

print(f"\n  ✅ Model berhasil dimuat dari checkpoint.")
print(f"  Path        : {checkpoint_v2_path}")
print(f"  Val MAE     : 0.8160 hari (best dari session sebelumnya)")
print(f"  Total params: {model_v2.count_params():,}")


CELL 2 — Arsitektur & Training LSTM

> ⚠️ **[EXPERIMENTAL/DEPRECATED — Tidak digunakan di pipeline final]**
>
> **Alasan:** V1 belum konvergen (berhenti di epoch 99 = max epoch, Val MAE=1.7516). V2 menghasilkan Val MAE=0.8160 di epoch 184 — improvement signifikan. V3 gagal (optimizer reset, epoch 5). V3 sudah dihapus dari notebook (PROJECT_LOG.md, 08E).
>
> **Nilai untuk presentasi:** Menunjukkan proses iteratif hyperparameter tuning dari V1 → V2 → V3 (gagal) → V2 sebagai final.
>
> **Status output:** Output sudah di-clear. Jalankan Cell 2 (load checkpoint) untuk melihat model tanpa re-training.

### 3. Arsitektur & Training LSTM V2
**Tujuan:** Mendefinisikan arsitektur LSTM V2 (2 LSTM layer + BatchNorm + Dropout + Dense) dan melatih ulang dari awal dengan max 200 epoch.  
**Input:** X_train_seq, y_train_seq, X_val_seq, y_val_seq (dari Cell 1)  
**Output:** models/dl_track/lstm_rul_best_v2.keras (checkpoint), history_v2 (training history)  
**Catatan:** HANYA JALANKAN JIKA ingin re-training. Waktu training: ~30-60 menit. Arsitektur V2: LSTM(64)+Dropout(0.3)+BatchNorm → LSTM(32)+Dropout(0.3)+BatchNorm → Dense(16) → Dense(1). Perubahan dari V1: dropout 0.2→0.3, +L2(0.001), epoch 100→200, patience 15→30.  


> [WARN] **[EXPERIMENTAL/DEPRECATED  Tidak digunakan di pipeline final]**
>
> **Alasan:** V1 belum konvergen (berhenti di epoch 99 = max epoch, Val MAE=1.7516). V2 menghasilkan Val MAE=0.8160 di epoch 184  improvement signifikan. V3 gagal (optimizer reset, epoch 5). V3 sudah dihapus dari notebook (PROJECT_LOG.md, 08E).
>
> **Nilai untuk presentasi:** Menunjukkan proses iteratif hyperparameter tuning dari V1  V2  V3 (gagal)  V2 sebagai final.
>
> **Status output:** Output sudah di-clear. Jalankan Cell 2 (load checkpoint) untuk melihat model tanpa re-training.

### 3. Arsitektur & Training LSTM V2
**Tujuan:** Mendefinisikan arsitektur LSTM V2 (2 LSTM layer + BatchNorm + Dropout + Dense) dan melatih ulang dari awal dengan max 200 epoch.  
**Input:** X_train_seq, y_train_seq, X_val_seq, y_val_seq (dari Cell 1)  
**Output:** models/dl_track/lstm_rul_best_v2.keras (checkpoint), history_v2 (training history)  
**Catatan:** HANYA JALANKAN JIKA ingin re-training. Waktu training: ~30-60 menit. Arsitektur V2: LSTM(64)+Dropout(0.3)+BatchNorm  LSTM(32)+Dropout(0.3)+BatchNorm  Dense(16)  Dense(1). Perubahan dari V1: dropout 0.20.3, +L2(0.001), epoch 100200, patience 1530.  


In [ ]:
# FASE 8E — Cell 2 (V2): Arsitektur & Training LSTM V2
from tensorflow.keras.regularizers import l2

print("=" * 65)
print("  FASE 8E — Cell 2: ARSITEKTUR & TRAINING LSTM V2")
print("=" * 65)

SEP = "=" * 65
sep = "-" * 65

LSTM_PARAMS_V2 = {
    "lstm_units_1"       : 64,
    "lstm_units_2"       : 32,
    "dropout_rate"       : 0.3,       # V1: 0.2 → lebih kuat
    "l2_reg"             : 0.001,     # V1: — → baru
    "learning_rate"      : 0.001,
    "batch_size"         : 32,
    "max_epochs"         : 200,       # V1: 100 → beri ruang konvergensi
    "patience"           : 30,        # V1: 15 → tunggu lebih lama
    "reduce_lr_factor"   : 0.5,
    "reduce_lr_patience" : 10,        # V1: 7 → LR decay lebih hati-hati
    "min_lr"             : 1e-6,
}

CHANGES = {
    "dropout_rate"       : "0.2 → 0.3  (regularisasi lebih kuat)",
    "l2_reg"             : "— → 0.001  (baru: L2 pada semua layer)",
    "max_epochs"         : "100 → 200  (beri ruang konvergensi)",
    "patience"           : "15 → 30    (tunggu lebih lama)",
    "reduce_lr_patience" : "7 → 10     (LR decay lebih hati-hati)",
}

print("\n  LSTM V2 Hyperparameters (+ keterangan perubahan dari V1):")
print(f"  {'-'*62}")
for k, v in LSTM_PARAMS_V2.items():
    change = f"  ← {CHANGES[k]}" if k in CHANGES else ""
    print(f"  {k:<24} : {str(v):<10}{change}")

# ────────────────────────────────────────────────────────────
# BANGUN ARSITEKTUR V2
# ────────────────────────────────────────────────────────────
print(f"\n{sep}")
print("  MEMBANGUN ARSITEKTUR V2")
print(sep)

input_shape = (X_train_seq.shape[1], X_train_seq.shape[2])
print(f"\n  Input shape : {input_shape}")

MODELS_DL_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path_v2 = str(MODELS_DL_DIR / "lstm_rul_best_v2.keras")

model_v2 = Sequential([
    LSTM(LSTM_PARAMS_V2["lstm_units_1"],
         input_shape=input_shape,
         return_sequences=True,
         kernel_regularizer=l2(LSTM_PARAMS_V2["l2_reg"]),
         recurrent_regularizer=l2(LSTM_PARAMS_V2["l2_reg"]),
         name="lstm_layer_1"),
    Dropout(LSTM_PARAMS_V2["dropout_rate"], name="dropout_1"),
    BatchNormalization(name="batch_norm_1"),

    LSTM(LSTM_PARAMS_V2["lstm_units_2"],
         return_sequences=False,
         kernel_regularizer=l2(LSTM_PARAMS_V2["l2_reg"]),
         recurrent_regularizer=l2(LSTM_PARAMS_V2["l2_reg"]),
         name="lstm_layer_2"),
    Dropout(LSTM_PARAMS_V2["dropout_rate"], name="dropout_2"),
    BatchNormalization(name="batch_norm_2"),

    Dense(16, activation="relu",
          kernel_regularizer=l2(LSTM_PARAMS_V2["l2_reg"]),
          name="dense_1"),
    Dense(1, activation="linear", name="output"),
])

model_v2.compile(
    optimizer=Adam(learning_rate=LSTM_PARAMS_V2["learning_rate"]),
    loss="mae",
    metrics=["mae", "mse"],
)

print(f"\n{sep}")
print("  MODEL V2 SUMMARY")
print(sep)
model_v2.summary()

# ────────────────────────────────────────────────────────────
# CALLBACKS V2
# ────────────────────────────────────────────────────────────
callbacks_v2 = [
    EarlyStopping(
        monitor="val_mae",
        patience=LSTM_PARAMS_V2["patience"],
        restore_best_weights=True,
        mode="min",
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=checkpoint_path_v2,
        monitor="val_mae",
        save_best_only=True,
        mode="min",
        verbose=0,
    ),
    ReduceLROnPlateau(
        monitor="val_mae",
        factor=LSTM_PARAMS_V2["reduce_lr_factor"],
        patience=LSTM_PARAMS_V2["reduce_lr_patience"],
        min_lr=LSTM_PARAMS_V2["min_lr"],
        mode="min",
        verbose=1,
    ),
]

# ────────────────────────────────────────────────────────────
# TRAINING V2
# ────────────────────────────────────────────────────────────
print(f"\n{sep}")
print("  TRAINING LSTM V2 (max 200 epochs, patience=30)...")
print("  Monitoring: val_mae | EarlyStopping aktif")
print(sep)

history_v2 = model_v2.fit(
    X_train_seq, y_train_seq,
    validation_data=(X_val_seq, y_val_seq),
    epochs=LSTM_PARAMS_V2["max_epochs"],
    batch_size=LSTM_PARAMS_V2["batch_size"],
    callbacks=callbacks_v2,
    verbose=1,
)

# ────────────────────────────────────────────────────────────
# RINGKASAN TRAINING
# ────────────────────────────────────────────────────────────
total_epochs  = len(history_v2.history["mae"])
best_val_mae  = float(min(history_v2.history["val_mae"]))
best_epoch    = int(np.argmin(history_v2.history["val_mae"])) + 1
lr_history    = history_v2.history.get("lr", [])
final_lr      = float(lr_history[-1]) if lr_history else LSTM_PARAMS_V2["learning_rate"]
converged     = total_epochs < LSTM_PARAMS_V2["max_epochs"]

V1_BEST_VAL_MAE = 1.7516
V1_BEST_EPOCH   = 99

print(f"\n{SEP}")
print("  TRAINING SELESAI — RINGKASAN V2")
print(SEP)
print(f"\n  Total epochs dijalankan : {total_epochs}")
print(f"  Best val_mae            : {best_val_mae:.4f} hari")
print(f"  Epoch terbaik           : {best_epoch}")
print(f"  Learning rate akhir     : {final_lr:.2e}")
print(f"  Early stopping          : {'✅ Aktif — konvergen' if converged else '⚠️ Mencapai max epoch'}")
print(f"  Checkpoint tersimpan    : {checkpoint_path_v2}")

print(f"\n{sep}")
print("  PERBANDINGAN V1 vs V2")
print(sep)
improvement = V1_BEST_VAL_MAE - best_val_mae
flag = "✅ Improvement" if improvement > 0 else "⚠️  Tidak ada improvement"
print(f"\n  V1 : best val_mae = {V1_BEST_VAL_MAE:.4f} hari di epoch {V1_BEST_EPOCH} (max epoch)")
print(f"  V2 : best val_mae = {best_val_mae:.4f} hari di epoch {best_epoch}")
print(f"  Delta (V1 - V2)   : {improvement:+.4f} hari  →  {flag}")

print(f"\n  [OK] Cell 2 selesai — lanjut ke Cell 3 untuk analisis history.")


CELL 3 — Training History Visualization

### 4. Training History Visualization
**Tujuan:** Memvisualisasikan learning curve (MAE full, MAE zoomed, Train-Val gap, Learning Rate schedule) untuk analisis konvergensi dan overfitting.  
**Input:** history_v2 (dari Cell 3)  ATAU gunakan Cell 2 checkpoint  
**Output:** Figure 2×2 panel: MAE Full, MAE Zoomed, Train-Val Gap, LR Schedule  
**Catatan:** Train-Val gap = 0.60 hari (moderate, acceptable). Val curve masih turun di akhir  bukan overfitting sejati. Root cause gap: Val set hanya 264 samples (statistical noise).  

[TARGET] **Dipakai di:** Paper IEEE Fig.5 (LSTM Training Curve) / Slide halaman 17

In [ ]:
# FASE 8E — Cell 3 (V2): Training History Analysis
print("=" * 65)
print("  FASE 8E — Cell 3: TRAINING HISTORY V2 ANALYSIS")
print("=" * 65)

SEP = "=" * 65
sep = "-" * 65

# ────────────────────────────────────────────────────────────
# HITUNG VARIABEL PENDUKUNG
# ────────────────────────────────────────────────────────────
mae_hist     = history_v2.history["mae"]
val_mae_hist = history_v2.history["val_mae"]
mse_hist     = history_v2.history.get("mse",     [])
val_mse_hist = history_v2.history.get("val_mse", [])
lr_hist      = history_v2.history.get("lr",      [])

gap           = [v - t for v, t in zip(val_mae_hist, mae_hist)]
total_ep      = len(mae_hist)
best_epoch_idx = int(np.argmin(val_mae_hist))   # 0-indexed untuk plot
best_epoch    = best_epoch_idx + 1              # 1-indexed untuk display
best_val      = float(min(val_mae_hist))
final_gap     = gap[-1] if gap else 0.0
epochs_range  = range(1, total_ep + 1)

# ────────────────────────────────────────────────────────────
# 2×2 FIGURE
# ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# PLOT 1 — MAE Full
ax = axes[0, 0]
ax.plot(epochs_range, mae_hist,     color="#2ecc71", linewidth=1.5,
        label="Train MAE")
ax.plot(epochs_range, val_mae_hist, color="#e74c3c", linewidth=1.5,
        label="Val MAE")
ax.axvline(best_epoch, color="gray", linestyle="--", linewidth=1.2,
           alpha=0.8, label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch", fontsize=10)
ax.set_ylabel("MAE (hari)", fontsize=10)
ax.set_title("Training History — MAE (Full)", fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# PLOT 2 — MAE Zoomed [0.5, 3.0]
ax = axes[0, 1]
ax.plot(epochs_range, mae_hist,     color="#2ecc71", linewidth=1.5,
        label="Train MAE")
ax.plot(epochs_range, val_mae_hist, color="#e74c3c", linewidth=1.5,
        label="Val MAE")
ax.axvline(best_epoch, color="gray", linestyle="--", linewidth=1.2,
           alpha=0.8, label=f"Best epoch ({best_epoch})")
ax.set_ylim([0.5, 3.0])
ax.set_xlabel("Epoch", fontsize=10)
ax.set_ylabel("MAE (hari)", fontsize=10)
ax.set_title("Training History — MAE (Zoomed: 0.5–3.0)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# PLOT 3 — Train-Val Gap
ax = axes[1, 0]
ax.plot(epochs_range, gap, color="#9b59b6", linewidth=1.5,
        label="Val MAE − Train MAE")
ax.axhline(0,   color="red",    linestyle="--", linewidth=1.2,
           alpha=0.8, label="Zero gap")
ax.axhline(0.5, color="orange", linestyle="--", linewidth=1.2,
           alpha=0.8, label="Gap warning threshold (0.5)")
ax.set_xlabel("Epoch", fontsize=10)
ax.set_ylabel("Val MAE − Train MAE", fontsize=10)
ax.set_title("Train-Val Gap per Epoch", fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# PLOT 4 — Learning Rate Schedule
ax = axes[1, 1]
if lr_hist:
    ax.plot(epochs_range, lr_hist, color="#e67e22", linewidth=1.5)
    ax.set_xlabel("Epoch", fontsize=10)
    ax.set_ylabel("Learning Rate", fontsize=10)
    ax.set_yscale("log")
    ax.set_title("Learning Rate Schedule", fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.3)
    # Annotate LR changes
    for i in range(1, len(lr_hist)):
        if lr_hist[i] < lr_hist[i - 1]:
            ax.annotate(f"LR↓ {lr_hist[i]:.2e}",
                        xy=(i + 1, lr_hist[i]),
                        xytext=(i + 3, lr_hist[i] * 1.5),
                        fontsize=7, color="red",
                        arrowprops=dict(arrowstyle="->", color="red",
                                        lw=0.8))
else:
    ax.set_facecolor("#f8f9fa")
    ax.text(0.5, 0.55,
            f"LR Schedule",
            ha="center", va="center", fontsize=13,
            fontweight="bold", color="#2c3e50",
            transform=ax.transAxes)
    ax.text(0.5, 0.40,
            f"0.001 → decay ×0.5 tiap 10 epoch stagnasi",
            ha="center", va="center", fontsize=10,
            color="#7f8c8d", transform=ax.transAxes,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="#bdc3c7", linewidth=1.2))
    ax.text(0.5, 0.27,
            f"min_lr = 1e-6",
            ha="center", va="center", fontsize=10,
            color="#7f8c8d", transform=ax.transAxes)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("Learning Rate Schedule", fontsize=11, fontweight="bold")
    for spine in ax.spines.values():
        spine.set_edgecolor("#bdc3c7")

plt.suptitle("LSTM V2 Training Analysis — RUL Predictor",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# ────────────────────────────────────────────────────────────
# INTERPRETASI OTOMATIS
# ────────────────────────────────────────────────────────────
print(f"\n{SEP}")
print("  INTERPRETASI LEARNING CURVE")
print(SEP)
print(f"\n  Total epochs       : {total_ep}")
print(f"  Best epoch         : {best_epoch}")
print(f"  Best Val MAE       : {best_val:.4f} hari")
print(f"  Final Train MAE    : {mae_hist[-1]:.4f} hari")
print(f"  Final Train-Val gap: {final_gap:.4f} hari")

# Konvergensi
if total_ep < LSTM_PARAMS_V2["max_epochs"]:
    print("\n  ✅ Early stopping aktif — model konvergen")
else:
    print("\n  ⚠️  Mencapai max epoch — belum konvergen penuh")

# Overfitting
if final_gap < 0.3:
    print("  ✅ Train-Val gap kecil — tidak overfitting")
elif final_gap < 0.8:
    print("  🟡 Train-Val gap sedang — slight overfit acceptable")
else:
    print("  🔴 Train-Val gap besar — overfit perlu diatasi")

# Perbaikan vs V1
V1_BEST_VAL_MAE = 1.7516
improvement     = V1_BEST_VAL_MAE - best_val
if improvement > 0.1:
    print(f"  ✅ V2 lebih baik dari V1 : +{improvement:.4f} hari improvement")
elif improvement > 0:
    print(f"  🟡 V2 sedikit lebih baik : +{improvement:.4f} hari")
else:
    print(f"  ⚠️  V2 tidak lebih baik dari V1 : {improvement:.4f} hari")

print(f"\n  [OK] Cell 3 selesai — lanjut ke Cell 4 untuk evaluasi.")


CELL 4 — Evaluasi & Comparison vs XGBoost

### 5. Evaluasi Final & Comparison vs XGBoost
**Tujuan:** Mengevaluasi LSTM V2 pada Val dan Test set, lalu membandingkan head-to-head dengan XGBoost Regressor.  
**Input:** model_v2 (dari Cell 2 atau 3), X_val_seq, y_val_seq, X_test_seq, y_test_seq  
**Output:** Metrik LSTM + Comparison table + Actual vs Predicted scatter plot  
**Catatan:** LSTM V2 unggul: MAE Test=0.7985 hari (vs XGBoost 1.1020), Error 1 hari=98.04% (vs 93.53%). LSTM dipilih sebagai Model 2 Final berdasarkan MAE Test terbaik dan Error  N hari.  

[TARGET] **Dipakai di:** Paper IEEE Table V (RUL Leaderboard) / Slide halaman 18

In [ ]:
# Gunakan model_v2 sebagai final model
model = model_v2

# FASE 8E — Cell 4: Evaluasi & Comparison vs XGBoost
print("=" * 65)
print("  FASE 8E — Cell 4: EVALUASI & COMPARISON vs XGBOOST")
print("=" * 65)

SEP = "=" * 65
sep = "-" * 65

# ────────────────────────────────────────────────────────────
# PREDIKSI & CLIP NEGATIF
# ────────────────────────────────────────────────────────────
y_val_pred  = np.maximum(model.predict(X_val_seq).flatten(),  0)
y_test_pred = np.maximum(model.predict(X_test_seq).flatten(), 0)

def compute_metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    pct1 = np.mean(np.abs(y_true - y_pred) <= 1.0) * 100
    pct3 = np.mean(np.abs(y_true - y_pred) <= 3.0) * 100
    return mae, rmse, r2, pct1, pct3

mae_val,  rmse_val,  r2_val,  pct1_val,  pct3_val  = compute_metrics(y_val_seq,  y_val_pred)
mae_test, rmse_test, r2_test, pct1_test, pct3_test = compute_metrics(y_test_seq, y_test_pred)

# ────────────────────────────────────────────────────────────
# TABEL METRIK LSTM
# ────────────────────────────────────────────────────────────
print(f"\n{sep}")
print("  METRIK LSTM (WARNING + CRITICAL ONLY)")
print(sep)
print(f"\n  {'Metrik':<24} {'Val':>10} {'Test':>10}")
print(f"  {'-'*47}")
for label, v, t in [
    ("MAE (hari)",         mae_val,  mae_test),
    ("RMSE (hari)",        rmse_val, rmse_test),
    ("R² Score",           r2_val,   r2_test),
    ("Error ≤ 1 hari (%)", pct1_val, pct1_test),
    ("Error ≤ 3 hari (%)", pct3_val, pct3_test),
]:
    print(f"  {label:<24} {v:>10.4f} {t:>10.4f}")

# ────────────────────────────────────────────────────────────
# COMPARISON vs XGBOOST
# ────────────────────────────────────────────────────────────
XGB_SCORES = {
    "MAE Val"         : 1.7189,
    "MAE Test"        : 1.1020,
    "RMSE Test"       : 5.6002,
    "R2 Test"         : 0.3961,
    "Error_1d_Test_%"  : 93.5335,
    "Error_3d_Test_%" : 94.9192,
}

def winner(xgb_v, lstm_v, higher_is_better=False):
    if higher_is_better:
        return "LSTM" if lstm_v > xgb_v else ("XGBoost" if xgb_v > lstm_v else "Tie")
    else:
        return "LSTM" if lstm_v < xgb_v else ("XGBoost" if xgb_v < lstm_v else "Tie")

print(f"\n{sep}")
print("  HEAD-TO-HEAD: XGBoost vs LSTM")
print(sep)

W = 58
print(f"\n  {'Metrik':<26} {'XGBoost':>10} {'LSTM':>10} {'Winner':>8}")
print(f"  {'-'*55}")

comparison_rows = [
    ("MAE Val (hari)",    XGB_SCORES["MAE Val"],          mae_val,  False),
    ("MAE Test (hari)",   XGB_SCORES["MAE Test"],         mae_test, False),
    ("RMSE Test",         XGB_SCORES["RMSE Test"],        rmse_test, False),
    ("R² Test",           XGB_SCORES["R2 Test"],          r2_test,   True),
    ("Error≤1hari Test%", XGB_SCORES["Error_1d_Test_%"],  pct1_test, True),
    ("Error≤3hari Test%", XGB_SCORES["Error_3d_Test_%"],  pct3_test, True),
]
for label, xgb_v, lstm_v, hib in comparison_rows:
    w = winner(xgb_v, lstm_v, hib)
    print(f"  {label:<26} {xgb_v:>10.4f} {lstm_v:>10.4f} {w:>8}")

diff = mae_test - XGB_SCORES["MAE Test"]
print(f"\n  Delta MAE Test (LSTM - XGBoost) : {diff:+.4f} hari")
print(f"\n  Verdict:")
if abs(diff) < 0.2:
    verdict = "🤝 Performa setara — pilih XGBoost (lebih simpel)"
elif mae_test < XGB_SCORES["MAE Test"]:
    verdict = "✅ LSTM lebih baik — sequence modeling efektif"
else:
    verdict = "📊 XGBoost lebih baik — tabular approach cukup"
print(f"  {verdict}")

# ────────────────────────────────────────────────────────────
# VISUALISASI ACTUAL vs PREDICTED
# ────────────────────────────────────────────────────────────
print(f"\n  Generating Actual vs Predicted plot...")

max_rul = max(float(y_test_seq.max()), float(y_test_pred.max()))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1 — LSTM
ax = axes[0]
ax.scatter(y_test_seq, y_test_pred, c="#9b59b6", alpha=0.5, s=20)
ax.plot([0, max_rul], [0, max_rul], "r--", linewidth=1.3,
        label="Perfect prediction")
ax.set_xlabel("Actual RUL (hari)", fontsize=10)
ax.set_ylabel("Predicted RUL (hari)", fontsize=10)
ax.set_title(f"LSTM — Actual vs Predicted\nMAE = {mae_test:.2f} hari",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Subplot 2 — XGBoost Referensi
ax = axes[1]
ax.set_facecolor("#f8f9fa")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.text(0.5, 0.55, "XGBoost Regressor",
        ha="center", va="center", fontsize=14,
        fontweight="bold", color="#2c3e50",
        transform=ax.transAxes)
ax.text(0.5, 0.40,
        f"MAE Test = {XGB_SCORES['MAE Test']:.4f} hari",
        ha="center", va="center", fontsize=13, color="#27ae60",
        transform=ax.transAxes,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor="#27ae60", linewidth=1.5))
ax.text(0.5, 0.25,
        f"Error ≤ 1 hari = {XGB_SCORES['Error_1d_Test_%']:.1f}%",
        ha="center", va="center", fontsize=11, color="#7f8c8d",
        transform=ax.transAxes)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title("XGBoost — Referensi\nMAE = 1.10 hari",
             fontsize=11, fontweight="bold")
for spine in ax.spines.values():
    spine.set_edgecolor("#bdc3c7")

plt.suptitle("RUL Predictor — Actual vs Predicted (Test Set)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\n  [OK] Cell 4 selesai — lanjut ke Cell 5 untuk export model.")


CELL 5 — Export Model

### 6. Export Model Final
**Tujuan:** Menyimpan model LSTM V2 final ke direktori models/dl_track sebagai artifact pipeline.  
**Input:** model_v2 (dari Cell 2 atau 3)  
**Output:** models/dl_track/lstm_rul_best_v2.keras (610.4 KB)  
**Catatan:** Model ini akan dipackage ulang di Fase 10 ke models/final/rul_predictor_final.keras.  


In [ ]:
# FASE 8E — Cell 5: Export Model
print("=" * 65)
print("  FASE 8E — Cell 5: EXPORT MODEL")
print("=" * 65)

SEP = "=" * 65
sep = "-" * 65

MODELS_DL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_PATH      = MODELS_DL_DIR / "lstm_rul_final.keras"
CHECKPOINT_PATH = MODELS_DL_DIR / "lstm_rul_best.keras"

model.save(str(FINAL_PATH))

final_mb      = FINAL_PATH.stat().st_size      / (1024 ** 2)
checkpoint_mb = CHECKPOINT_PATH.stat().st_size  / (1024 ** 2) \
    if CHECKPOINT_PATH.exists() else 0.0

total_params   = model.count_params()
best_epoch_val = min(history_v2.history["val_mae"])

print(f"\n  Path lstm_rul_final.keras : {FINAL_PATH}")
print(f"  Ukuran final              : {final_mb:.2f} MB")
print(f"  Path lstm_rul_best.keras  : {CHECKPOINT_PATH}")
print(f"  Ukuran checkpoint         : {checkpoint_mb:.2f} MB")
print(f"  Total params model        : {total_params:,}")
print(f"  Best val_mae              : {best_epoch_val:.4f} hari")

print(f"\n{sep}")
print("  RINGKASAN PERFORMA AKHIR")
print(sep)
print(f"\n  MAE  (Val)            : {mae_val:.4f} hari")
print(f"  MAE  (Test)           : {mae_test:.4f} hari")
print(f"  RMSE (Val)            : {rmse_val:.4f} hari")
print(f"  RMSE (Test)           : {rmse_test:.4f} hari")
print(f"  R²   (Val)            : {r2_val:.4f}")
print(f"  R²   (Test)           : {r2_test:.4f}")
print(f"  Error ≤ 1 hari (Test) : {pct1_test:.2f}%")
print(f"  Error ≤ 3 hari (Test) : {pct3_test:.2f}%")
print(f"  Scope                 : WARNING + CRITICAL only")

gap = abs(mae_val - mae_test)
flag = "[OK] Stabil" if gap < 2.0 else "[WARN] Gap > 2 hari"
print(f"  Gap Val vs Test (MAE) : {gap:.4f} hari  →  {flag}")

print(f"\n{sep}")
print("  CATATAN DEPLOYMENT")
print(sep)
print("\n  Model ini hanya digunakan saat Model 1 mendeteksi")
print("  status WARNING atau CRITICAL.")
print("  Tidak digunakan saat status HEALTHY.")
print("\n  Fair comparison: data identik dengan XGBoost 08D")
print(f"  (WARNING+CRITICAL only, SEQ_LEN={SEQ_LEN})")
print("\n  Inference flow:")
print("  [Sensor Data] → reshape ke (1, SEQ_LEN, 69)")
print("      → model.predict() → clip ke 0 → rul_days")

print(f"\n{SEP}")
print("  [OK] LSTM RUL Predictor — Fase 8E SELESAI.")
print(f"  [OK] Final model : lstm_rul_final.keras  ({final_mb:.2f} MB)")
print(f"  [OK] Best model  : lstm_rul_best.keras   ({checkpoint_mb:.2f} MB)")
print(SEP)
